In [ ]:
import numpy as np
import pandas as pd

In [ ]:
match=pd.read_csv('matches.csv')
delivery=pd.read_csv('deliveries.csv')

In [ ]:
match.tail()

In [ ]:
delivery.tail()


Now for Regression Model we need data like batting team(in 2nd innings) ,bolwing team,runs to be chased in required number of balls,match id

In [ ]:
match['team1'].unique()

In [ ]:
teams=[
    'Royal Challengers Bengaluru',
    'Mumbai Indians',
    'Kolkata Knight Riders',
    'Rajasthan Royals',
    'Chennai Super Kings',
    'Sunrisers Hyderabad',
    'Lucknow Super Giants',
    'Gujarat Titans',
    'Delhi Capitals',
    'Punjab Kings'
]
    

In [ ]:
match['team1']=match['team1'].str.replace('Royal Challengers Bangalore','Royal Challengers Bengaluru')
match['team2']=match['team2'].str.replace('Royal Challengers Bangalore','Royal Challengers Bengaluru')

match['team1']=match['team1'].str.replace('Kings XI Punjab','Punjab Kings')
match['team2']=match['team2'].str.replace('Kings XI Punjab','Punjab Kings')

match['team1']=match['team1'].str.replace('Deccan Chargers','Sunrisers Hyderabad')
match['team2']=match['team2'].str.replace('Deccan Chargers','Sunrisers Hyderabad')

match['team1']=match['team1'].str.replace('Gujarat Lions','Gujarat Titans')
match['team2']=match['team2'].str.replace('Gujarat Lions','Gujarat Titans')

match['team1']=match['team1'].str.replace('Delhi Daredevils','Delhi Capitals')
match['team2']=match['team2'].str.replace('Delhi Daredevils','Delhi Capitals')



In [ ]:
match=match[match['team1'].isin(teams)]
match=match[match['team2'].isin(teams)]

In [ ]:
match['team1'].unique()

In [ ]:
delivery=delivery[delivery['inning']==1]

In [ ]:
match['winner'] = match.apply(lambda row: row['team1'] if "runs" in row['result'] 
                                    else (row['team2'] if "wickets" in row['result'] 
                                          else "Draw/Tie"), axis=1)

In [ ]:
match = match[['id', 'city', 'winner', 'target_runs']]

In [ ]:
match=match.merge(delivery,left_on='id',right_on='match_id')

In [ ]:
match


Here Total_runs_y colunm is in string format so We have to convert into int format.


In [ ]:
match['total_runs_y'] = pd.to_numeric(match['total_runs'], errors='coerce')

Now we will calculate cumulative score per each

In [ ]:
match['cumulative_runs'] = match.groupby('id')['total_runs'].cumsum()

In [ ]:
match

In [ ]:
match['runs_left']=match['target_runs']-match['cumulative_runs']

In [ ]:
 match['balls_left']=120-(match['over']*6+match['ball'])

Here We Are calculating Wickets According to Score

In [ ]:
match['is_wicket'] = pd.to_numeric(match['is_wicket'], errors='coerce').fillna(0).astype(int)

In [ ]:
match['wickets'] = 10 - match.groupby('id')['is_wicket'].cumsum()


In [ ]:
match

In [ ]:
match['crr']=(match['cumulative_runs']*6)/(120-match['balls_left'])

In [ ]:
match['rrr']=(match['runs_left']*6)/(match['balls_left'])

In [ ]:
def result(row):
    return 1 if row['batting_team'] == row['winner'] else 0

In [ ]:
match['result'] = match.apply(result,axis=1)

For Regression Model We Need batting_team,bowling_team,city,runs_left,balls_left,wickets,target,crr,rrr,result

In [ ]:
final_df = match[['batting_team','bowling_team','city','runs_left','balls_left','wickets','target_runs','crr','rrr','result']]

In [ ]:
final_df

In [ ]:
final_df = final_df[final_df['balls_left'] != 0]

In [ ]:
final_df = final_df.dropna()


Regression

In [ ]:
X = final_df.iloc[:,:-1]
y = final_df.iloc[:,-1]
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

trf = ColumnTransformer([
    ('trf', OneHotEncoder(sparse_output=False, drop='first'), ['batting_team', 'bowling_team', 'city'])
], remainder='passthrough',
    force_int_remainder_cols=False)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipe = Pipeline(steps=[
    ('step1',trf),
    ('step2',LogisticRegression(solver='liblinear'))
])

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
y_pred = pipe.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

In [ ]:
import pickle
pickle.dump(pipe,open('temp.pkl','wb'))